# Interface web Gradio pour GitHub Issues

Ces travaux pratiques partent de la solution des travaux pratiques *Interface en ligne de commande pour GitHub Issues* : `ghcli list` et `ghcli create` listent et créent les tickets d'un dépôt, et sont testés. Nous allons leur ajouter une interface web, avec Gradio.

Le code est celui de la branche `gradio` du dépôt [shuuchuu/ghcli](https://github.com/shuuchuu/ghcli/tree/gradio), que le notebook clone. `src/ghcli/gui.py` est l'interface à compléter : `construit_app` construit une application à deux onglets, vides pour l'instant, et `_formate_ticket` écrit un ticket en Markdown. Un fichier s'écrit avec une cellule qui commence par `%%writefile <fichier>`, qui le remplace par le reste de son contenu, ou dans l'explorateur de fichiers, à gauche (un double clic l'ouvre dans un éditeur). Les solutions sont aussi sur la [branche `gradio-solution`](https://github.com/shuuchuu/ghcli/tree/gradio-solution) du dépôt.

## Mise en place de l'environnement de travail

Exécutez la cellule suivante : elle clone le dépôt dans le dossier `ghcli/`, installe ses dépendances, et rend son paquet importable.

In [ ]:
import pathlib
import sys

dossier_racine = pathlib.Path.cwd()
if dossier_racine.name != "ghcli":
  dossier_racine /= "ghcli"
if not dossier_racine.exists():
  !git clone -b gradio https://github.com/shuuchuu/ghcli.git {dossier_racine}
%cd {dossier_racine}
!uv pip install gradio niquests
if str(dossier_racine / "src") not in sys.path:
  sys.path.insert(0, str(dossier_racine / "src"))

`ghcli` a besoin d'un jeton GitHub : si vous n'en avez pas, créez un dépôt de test et un jeton comme l'explique [l'énoncé de l'interface en ligne de commande](https://github.com/shuuchuu/ghcli#un-dépôt-de-test-et-un-jeton-github). Écrivez-le dans la cellule suivante, entre les guillemets, puis exécutez-la.

In [ ]:
import os

jeton = ""
os.environ["GHCLI_TOKEN"] = jeton

La cellule suivante lance l'interface : Colab l'affiche sous la cellule, et donne un lien public (`*.gradio.live`) qui l'ouvre dans un autre onglet. Relancez-la après chaque modification de `gui.py` : elle relit le fichier.

In [ ]:
import importlib

import ghcli.gui

# Relit gui.py, après une modification.
importlib.reload(ghcli.gui)
ghcli.gui.construit_app().launch()

## Lister les tickets d'un dépôt

Dans l'onglet *Lister les tickets* de `construit_app`, ajoutez deux zones de texte, le propriétaire du dépôt et son nom, un bouton *Afficher les tickets* et une zone de Markdown où s'affichent les tickets. Le bouton appelle une fonction `_liste_tickets(proprietaire: str, depot: str) -> str`, à écrire, qui renvoie les tickets du dépôt en Markdown, séparés par une ligne horizontale (`---`), `Aucun ticket.` s'il n'en a pas, ou un message d'erreur si l'API en renvoie une.

**Documentation utile**

- https://www.gradio.app/guides/blocks-and-event-listeners
- https://www.gradio.app/docs/gradio/textbox
- https://www.gradio.app/docs/gradio/markdown

In [ ]:
# Votre code ici

### Conseils supplémentaires

Dans un bloc `with gr.Tab(...)`, chaque composant créé s'ajoute à l'onglet, et `with gr.Row():` en place plusieurs côte à côte. `bouton.click(fonction, inputs=[...], outputs=...)` appelle la fonction avec les valeurs des composants d'entrée, et affiche ce qu'elle renvoie dans celui de sortie. `liste_tickets` lève une `niquests.HTTPError` pour un dépôt inconnu.

```python
def _liste_tickets(proprietaire: str, depot: str) -> str:
  try:
    tickets = liste_tickets(proprietaire, depot)
  except HTTPError as erreur:
    return ...
  return "\n\n---\n\n".join(...) or "Aucun ticket."


# Dans construit_app :
with gr.Tab("Lister les tickets"):
  with gr.Row():
    proprietaire = gr.Textbox(label="Propriétaire", placeholder="octocat")
    ...
  bouton = gr.Button("Afficher les tickets")
  tickets = gr.Markdown()
  bouton.click(_liste_tickets, inputs=[...], outputs=...)
```

### Solution

In [ ]:
%%writefile src/ghcli/gui.py
"""Interface web pour les tickets GitHub, avec Gradio."""

import gradio as gr
from niquests import HTTPError

from .api import liste_tickets
from .modele import Ticket


def _formate_ticket(ticket: Ticket) -> str:
    """Montre un ticket en Markdown : son titre, lié à sa page, puis son corps.

    Args:
        ticket: Le ticket à montrer.

    Returns:
        Son Markdown.
    """
    return f"### [{ticket.titre}]({ticket.url})\n\n{ticket.corps}"


def _liste_tickets(proprietaire: str, depot: str) -> str:
    try:
        tickets = liste_tickets(proprietaire, depot)
    except HTTPError as erreur:
        return f"Impossible de lister les tickets : {erreur}"
    return "\n\n---\n\n".join(map(_formate_ticket, tickets)) or "Aucun ticket."


def construit_app() -> gr.Blocks:
    """Construit l'application : un onglet liste les tickets, un autre en crée un.

    Returns:
        L'application, que `launch()` sert.
    """
    with gr.Blocks(title="ghcli") as app:
        gr.Markdown("# ghcli\n\nListe et crée les tickets d'un dépôt GitHub.")
        with gr.Tab("Lister les tickets"):
            with gr.Row():
                proprietaire = gr.Textbox(label="Propriétaire", placeholder="octocat")
                depot = gr.Textbox(label="Dépôt", placeholder="Hello-World")
            bouton = gr.Button("Afficher les tickets")
            tickets = gr.Markdown()
            # Les arguments de la fonction sont les valeurs des entrées, sa valeur de
            # retour celle de la sortie.
            bouton.click(_liste_tickets, inputs=[proprietaire, depot], outputs=tickets)
        with gr.Tab("Créer un ticket"):
            # Votre code ici
            ...
    return app


def cree_interface() -> None:
    """Sert l'interface, à l'adresse http://127.0.0.1:7860 par défaut."""
    construit_app().launch()

In [ ]:
import importlib

import ghcli.gui

# Relit gui.py, après une modification.
importlib.reload(ghcli.gui)
ghcli.gui.construit_app().launch()

## Créer un ticket

Dans l'onglet *Créer un ticket*, ajoutez le propriétaire du dépôt, son nom, le titre du ticket et son corps (une zone de texte de plusieurs lignes), un bouton *Créer le ticket* et une zone de Markdown qui affiche le ticket créé, ou un message d'erreur. Le bouton appelle une fonction `_cree_ticket`, à écrire. Créez un ticket dans votre dépôt de test, puis listez ses tickets dans l'autre onglet.

In [ ]:
# Votre code ici

### Conseils supplémentaires

`gr.Textbox(label="Corps", lines=6)` est une zone de texte de six lignes. Les deux onglets peuvent réutiliser les mêmes noms de variables : chaque `click` reçoit les composants qui existent à son appel.

```python
def _cree_ticket(proprietaire: str, depot: str, titre: str, corps: str) -> str:
  try:
    ticket = cree_ticket(proprietaire, depot, titre, corps)
  except HTTPError as erreur:
    return ...
  return _formate_ticket(ticket)
```

### Solution

In [ ]:
%%writefile src/ghcli/gui.py
"""Interface web pour les tickets GitHub, avec Gradio."""

import gradio as gr
from niquests import HTTPError

from .api import cree_ticket, liste_tickets
from .modele import Ticket


def _formate_ticket(ticket: Ticket) -> str:
    """Montre un ticket en Markdown : son titre, lié à sa page, puis son corps.

    Args:
        ticket: Le ticket à montrer.

    Returns:
        Son Markdown.
    """
    return f"### [{ticket.titre}]({ticket.url})\n\n{ticket.corps}"


def _liste_tickets(proprietaire: str, depot: str) -> str:
    try:
        tickets = liste_tickets(proprietaire, depot)
    except HTTPError as erreur:
        return f"Impossible de lister les tickets : {erreur}"
    return "\n\n---\n\n".join(map(_formate_ticket, tickets)) or "Aucun ticket."


def _cree_ticket(proprietaire: str, depot: str, titre: str, corps: str) -> str:
    try:
        ticket = cree_ticket(proprietaire, depot, titre, corps)
    except HTTPError as erreur:
        return f"Impossible de créer le ticket : {erreur}"
    return _formate_ticket(ticket)


def construit_app() -> gr.Blocks:
    """Construit l'application : un onglet liste les tickets, un autre en crée un.

    Returns:
        L'application, que `launch()` sert.
    """
    with gr.Blocks(title="ghcli") as app:
        gr.Markdown("# ghcli\n\nListe et crée les tickets d'un dépôt GitHub.")
        with gr.Tab("Lister les tickets"):
            with gr.Row():
                proprietaire = gr.Textbox(label="Propriétaire", placeholder="octocat")
                depot = gr.Textbox(label="Dépôt", placeholder="Hello-World")
            bouton = gr.Button("Afficher les tickets")
            tickets = gr.Markdown()
            # Les arguments de la fonction sont les valeurs des entrées, sa valeur de
            # retour celle de la sortie.
            bouton.click(_liste_tickets, inputs=[proprietaire, depot], outputs=tickets)
        with gr.Tab("Créer un ticket"):
            with gr.Row():
                proprietaire = gr.Textbox(label="Propriétaire")
                depot = gr.Textbox(label="Dépôt")
            titre = gr.Textbox(label="Titre")
            corps = gr.Textbox(label="Corps", lines=6)
            bouton = gr.Button("Créer le ticket")
            ticket = gr.Markdown()
            bouton.click(
                _cree_ticket, inputs=[proprietaire, depot, titre, corps], outputs=ticket
            )
    return app


def cree_interface() -> None:
    """Sert l'interface, à l'adresse http://127.0.0.1:7860 par défaut."""
    construit_app().launch()

In [ ]:
import importlib

import ghcli.gui

# Relit gui.py, après une modification.
importlib.reload(ghcli.gui)
ghcli.gui.construit_app().launch()